# Can we predict room occupancy?
Target: `Occupancy` (0/1) from environmental sensors (Temperature, Humidity, Light, CO2, HumidityRatio).

Rows are ordered in time, so everything is evaluated **chronologically**: models train on the earlier part of the data and are tested on the later part. A random split would leak neighbouring minutes (nearly identical readings) between train and test and give over-optimistic scores.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)
SEED=0
df=pd.read_csv('data.csv',parse_dates=['date'])
print(df.shape, 'missing:',df.isna().sum().sum(), '| sorted:',df.date.is_monotonic_increasing)
df.describe().T

## 1. Data overview

In [ ]:
gap=df.date.diff().dt.total_seconds()
print(df.date.min(), '->', df.date.max())
print('gaps > 2 min:'); print(df.loc[gap>120,'date'].to_frame().assign(gap_hours=gap[gap>120]/3600))
print(df.Occupancy.value_counts(normalize=True).rename('share'))
daily=df.groupby(df.date.dt.date).Occupancy.agg(['mean','size'])
daily['weekday']=pd.to_datetime(daily.index).day_name()
daily

In [ ]:
fig,ax=plt.subplots(5,1,figsize=(13,10),sharex=True)
for a,col in zip(ax,['Temperature','Humidity','Light','CO2','HumidityRatio']):
    a.plot(df.date,df[col],lw=.5); a.set_ylabel(col)
    a.fill_between(df.date,df[col].min(),df[col].max(),where=df.Occupancy==1,color='orange',alpha=.25,step='mid')
ax[0].set_title('Sensor readings (orange = occupied)'); plt.tight_layout(); plt.show()

In [ ]:
feats=['Temperature','Humidity','Light','CO2','HumidityRatio']
fig,ax=plt.subplots(1,5,figsize=(16,3))
for a,col in zip(ax,feats):
    for k,g in df.groupby('Occupancy'): a.hist(g[col],bins=40,alpha=.6,density=True,label=f'occ={k}')
    a.set_title(col)
ax[0].legend(); plt.tight_layout(); plt.show()
df[feats+['Occupancy']].corr()['Occupancy'].drop('Occupancy')

## 2. Chronological split
First 70% of rows for training/model selection, last 30% as a held-out test set that is only touched once at the end. Model selection uses expanding-window `TimeSeriesSplit` inside the training part.

In [ ]:
n=len(df); cut=int(n*.7)
train,test=df.iloc[:cut],df.iloc[cut:]
print('train',train.date.min(),'->',train.date.max(),'| occ rate %.3f'%train.Occupancy.mean())
print('test ',test.date.min(),'->',test.date.max(),'| occ rate %.3f'%test.Occupancy.mean())

## 3. Feature sets
* **sensors** – the five raw readings.
* **sensors + past context** – adds only *backward-looking* features (previous-reading differences and trailing rolling means), so nothing from the future is used. This assumes consecutive readings, which holds apart from the two gaps shown above.

Hour-of-day is deliberately left out: it would let a model learn the office schedule rather than sense occupancy.

In [ ]:
def add_context(d):
    d=d.copy()
    for col in ['Light','CO2','Temperature']:
        d[col+'_diff1']=d[col].diff()
        d[col+'_roll10']=d[col].rolling(10,min_periods=1).mean()
    d['CO2_diff10']=d['CO2'].diff(10)
    return d.bfill()   # only the first few rows are affected
full=add_context(df)
sets={'sensors':feats,
      'sensors + past context':[c for c in full.columns if c not in ('date','Occupancy')]}
Xtr,ytr=full.iloc[:cut],df.Occupancy.iloc[:cut]
Xte,yte=full.iloc[cut:],df.Occupancy.iloc[cut:]

## 4. Model comparison (time-series CV on the training part)

In [ ]:
models={
 'majority baseline':DummyClassifier(strategy='most_frequent'),
 'logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'random forest':RandomForestClassifier(300,min_samples_leaf=5,n_jobs=-1,random_state=SEED),
 'hist. gradient boosting':HistGradientBoostingClassifier(random_state=SEED),
}
tscv=TimeSeriesSplit(5)
rows=[]
for sname,cols in sets.items():
    for mname,m in models.items():
        r={'features':sname,'model':mname}
        for met in ['accuracy','f1','roc_auc']:
            r[met]=cross_val_score(m,Xtr[cols],ytr,cv=tscv,scoring=met).mean()
        rows.append(r)
cv=pd.DataFrame(rows).round(3); cv

## 5. Held-out test evaluation

In [ ]:
def evaluate(m,cols):
    m.fit(Xtr[cols],ytr); p=m.predict_proba(Xte[cols])[:,1]; pred=(p>.5).astype(int)
    return p,pred,dict(accuracy=accuracy_score(yte,pred),f1=f1_score(yte,pred),
                       roc_auc=roc_auc_score(yte,p),pr_auc=average_precision_score(yte,p))
res={}; rows=[]
for sname,cols in sets.items():
    for mname,m in models.items():
        p,pred,s=evaluate(m,cols); res[(sname,mname)]=(p,pred)
        rows.append({'features':sname,'model':mname,**s})
test_tab=pd.DataFrame(rows).round(3); test_tab

In [ ]:
best=('sensors','random forest')
p,pred=res[best]
print(best); print(classification_report(yte,pred,target_names=['empty','occupied'],digits=3))
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
cm=confusion_matrix(yte,pred); ax[0].imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax[0].text(j,i,cm[i,j],ha='center',va='center')
ax[0].set(xticks=[0,1],yticks=[0,1],xticklabels=['empty','occupied'],yticklabels=['empty','occupied'],xlabel='predicted',ylabel='actual',title='Confusion matrix')
ax[1].plot(test.date,yte,lw=1,label='actual'); ax[1].plot(test.date,p,lw=.6,label='P(occupied)'); ax[1].legend(); ax[1].set_title('Test period')
plt.tight_layout(); plt.show()

### Where does it go wrong, and what drives predictions?

In [ ]:
err=test.assign(pred=pred,p=p)[pred!=yte.values]
print('errors:',len(err),'of',len(test))
print(err.groupby(err.date.dt.date).size().rename('errors per day'))
rf=models['random forest'].fit(Xtr[feats],ytr)
pd.Series(rf.feature_importances_,feats).sort_values().plot.barh(title='Random forest importance (sensors only)'); plt.show()

### Robustness: does it work on a different day?
Walk-forward by whole day: train on all earlier days, predict the next day (days with at least 500 readings).

In [ ]:
days=full.assign(day=df.date.dt.date,y=df.Occupancy)
order=sorted(days.day.unique()); out=[]
for i,d in enumerate(order):
    if i<4: continue
    tr,te=days[days.day<d],days[days.day==d]
    if len(te)<500: continue
    m=RandomForestClassifier(300,min_samples_leaf=5,n_jobs=-1,random_state=SEED).fit(tr[feats],tr.y)
    pr=m.predict(te[feats])
    out.append({'day':d,'weekday':pd.Timestamp(d).day_name(),'occ_rate':te.y.mean(),'accuracy':accuracy_score(te.y,pr)})
pd.DataFrame(out).round(3)

## 6. Conclusions
Yes — occupancy is highly predictable from the sensors, even when evaluated forward in time.

* The majority-class baseline gets ~77% accuracy (≈82% on the test period); the models reach **~98–99% accuracy, F1 ≈ 0.96–0.97, ROC-AUC ≈ 0.99+** on the held-out last 30% of the data (random forest, sensors only: accuracy 0.987, occupied recall 0.997, precision 0.932).
* **Light** is by far the strongest single signal (correlation ≈ 0.91 with occupancy); CO2 and Temperature add information. Even plain logistic regression is nearly as good as tree ensembles, so the signal is simple. Backward-looking context features give little to no consistent gain.
* Errors are concentrated in a few periods: most test errors (60 of 83) fall on Sunday 2015-02-15, where the room was empty but readings looked occupied (false positives), which explains the lower precision. A model trained on weekdays has seen little of that kind of weekend/idle-but-lit condition.

**Caveats:** only ~16 days from one room, so the test period is just 4 days and weekend behaviour is barely represented; light may be a proxy that breaks if lighting habits change (e.g. lights left on in an empty room); and the labels are assumed to be ground truth. Collecting more weeks (and validating in other rooms/seasons) would be needed before relying on this in production.

*Note: the notebook was saved without cell outputs because the sandbox could not start a Jupyter kernel; the code was verified by running every cell directly in Python, and the numbers above come from that run. Re-run all cells to populate outputs.*